# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnzilaAhsan/week1-asm1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring** (Lane 2).

Question: Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

I have selected this lane as it gives a result of an actionable ranking of items along with reason codes, as opposed to merely identifying correlation (Lane 1) or developing a typology without urgency (Lane 3). The rationale behind the choice is to get practical experience in applying the whole lane technique: having an interpretable baseline ranking first, then making sure logistic regression, decision tree, random forest or gradient boosting model justify their added complexity versus baseline, producing a ranking with reason codes and performing an honest evaluation with precision@K, recall and average precision metrics as well as manual inspection of top 20 items. Using Lane 1 as a backup if this lane fails in Week 3 leakage test.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Research question: Of all the content items of a client, which ones should the content editor analyze for refreshing during this week, and in what order?

Unit of analysis: One content item (one page) that is scored independently in the context of its client.

Decision it affects: Which page(s) an editor spends his/her scarce hours on first, from potentially thousands of pages per client.

Actor and what he does: Content editor or SEO lead of a FlyRank client. Having a ranked list of items, the editor opens them one by one and decides whether to re-write, to enlarge, to cut or do nothing – an actionable decision rather than a number.

Output: Ranked list of items (content_id, score, reason_code, action).

Cost of a false positive:

We flag a page as urgent and we are wrong, but it was not declining or did not need refreshing. Cost = wasted editor time. The true cost is opportunity cost because editor time is our most precious resource in this process.
False negative: a page loses its traffic, and we don't discover that. Slow, gradual loss of impression/clicks — hard to notice, but more expensive in the long run. Since the cost of a false positive is wasted time and not broken something, I don't need a highly conservative threshold: it is perfectly alright for the queue to have a lot of borderline candidates, as long as the order (top-urgency-first) is reliable.
Why data/ML would help here: with 30,000 entries and several interacting signals (position, trend, freshness, impressions, content type), each of which is a factor in deciding whether a page needs a refresh and in estimating its urgency, the rule "if declining, refresh" is too simplistic. An easily interpretable baseline score derived from several signals will rank pages better than looking at individual columns, even though it is closer to a scoring/ranking problem than a machine learning one.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

import pandas as pd
import os

# Works locally (running from work/notebooks/) and in Colab (repo not cloned yet)
local_path = "../../data/raw/content_refresh_anonymized.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
else:
    # Colab: clone the repo once, then read from it
    if not os.path.exists("week1-asm1"):
        !git clone https://github.com/UnzilaAhsan/internship.git
    df = pd.read_csv("week1-asm1/data/raw/content_refresh_anonymized.csv")

n = len(df)
print(f"Rows: {n:,}  |  Clients: {df['client_id'].nunique()}")


# 1) How much of the inventory is trending down at all?
down = df[df["trend_direction"] == "down"]
print(f"1) Rows trending down: {len(down):,} ({len(down)/n*100:.1f}% of all rows)")

# 2) Of those, how many combine a decline with real, measurable demand
#    (not just noise on a page nobody sees) and a position worth defending (top 20)?
#    avg_position == 0 means "no data", so it's excluded, not treated as rank 0.
decay_candidates = df[
    (df["trend_direction"] == "down")
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["impressions_90d"] > 50)
]
print(f"2) Declining + ranked top-20 + impressions_90d>50: {len(decay_candidates):,} "
      f"({len(decay_candidates)/n*100:.1f}%)")

# 3) A tighter, higher-priority slice: pages in striking distance (positions ~11-20,
#    i.e. page 2) that are trending down. These are the pages closest to a page-1
#    breakthrough that are instead sliding backward -- a natural "fix first" candidate.
striking_down = df[(df["position_tier"] == "striking") & (df["trend_direction"] == "down")]
print(f"3) 'Striking distance' pages trending down: {len(striking_down):,} "
      f"({len(striking_down)/n*100:.1f}%)")

# 4) Staleness context: how much of the inventory hasn't been touched in a long time
#    AND is still getting impressions (i.e. worth touching)?
stale_visible = df[df["freshness_tier"].isin(["91-180", "181+"]) & (df["impressions_last_30d"] > 0)]
print(f"4) Stale (90+ days) but still getting impressions: {len(stale_visible):,} "
      f"({len(stale_visible)/n*100:.1f}%)")


Rows: 30,000  |  Clients: 32
1) Rows trending down: 16,262 (54.2% of all rows)
2) Declining + ranked top-20 + impressions_90d>50: 9,967 (33.2%)
3) 'Striking distance' pages trending down: 4,452 (14.8%)
4) Stale (90+ days) but still getting impressions: 8,741 (29.1%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim**

Observation: "54.2% of rows in the starter sample are marked down in trend_direction" - a characteristic of the data set, rather than an absolute statement on people's search habits.
Direction: "Pages that have been declining and are still getting some impression traffic (impressions_90d > 50) represent one third of the sample, and could form a refresh queue" – a plausible direction, but not necessarily a cause-and-effect relationship.
Decision support: Ultimately, the end result is a ranked queue to assist a person in making decisions about the pages.


**What I will never claim:**

The reason for recovery after refreshing a page is observational evidence – I don't have access to any counterfactuals (the situation if the refresh did not happen), and thus cannot make causal claims without performing a before-and-after or holdout experiment.
The assumption of prediction of Google ranking or "of how Google's algorithm works". I only have FlyRank's own search/analytics metrics, no internals of Google.
The trend_direction and trend_pct are "signals" detected – they were precalculated according to the same logic used to label decline instances, and therefore using them as a signal would create a logical tautology. I'll use them solely as label data, not as evidence of a discovered signal.
A wrong decision when a score is too high can be very costly – as was already explained above, the cost of a false positive in this situation is low (lost time reviewing the website), and therefore the scoring/ranking approach is justified.

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.